# CDC-CP P1d: three backbones, three seeds each

This notebook picks the P2 backbone by **unseen disease Top-1**, the metric we chose after we found that the calibrated H mostly measures crop knowledge.

| Backbone | Head | Seeds |
|---|---|---|
| EfficientNet-B0 | 256, ReLU | 42, 43, 44 |
| ResNet50 | 256, ReLU | 42, 43, 44 |
| ConvNeXt-Small (new) | 256, ReLU | 42, 43, 44 |

Every run uses the P1 recipe `configs/p1_lr1e-4.yaml`: backbone LR 1e-4, heads 1e-3, 20 epochs, batch 32, Baseline 1 augmentation. Only the backbone and the seed change.

**Pick rule (`cdccp/selection.py: pick_backbone`, unit-tested).** The winner has the highest mean unseen disease Top-1 over the 3 seeds. Each run counts its last 3 epochs. If the gap to the runner-up is no more than 2 standard errors of the difference, the notebook says **TIE**. That error counts both the seed spread and the noise of scoring only 400 unseen images, so with today's numbers a win needs a gap of about 6 points. Then the choice is ours to make, not the notebook's. Keep-seen H and seen disease are shown for the record.

**Setup (once):** inputs are the code dataset (the new `cdc-cp_code.zip`) and the data dataset, as before. Optional: add the old P1 sweep notebook's output as an input. Its seed-42 runs of B0 and ResNet50 are then reused, which saves two runs. **Accelerator → GPU T4 x2**, **Internet → On**.

**First, a smoke test:** keep `SMOKE = True` and run all (about 10 min). Then set `SMOKE = False` and **Save Version → Save & Run All (Commit)**.

**If it stops early:** add this notebook's last version (its output) as an input, then Save & Run All again. Finished runs are skipped and an interrupted run resumes.

Only `DPD_Small_train.csv` and its 8,000 images are read. The official test files are never opened.

## 1. Settings

In [ ]:
from pathlib import Path

SMOKE    = True    # True: 1 epoch of each backbone (seed 42) to check it builds on the GPU. Then set False and commit.
TOL      = 3.0     # keep-seen: seen accuracy may fall at most this many points below raw (shown for the record)
WORKERS  = 2       # data-loader workers per run (2 runs share Kaggle's 4 CPU cores)

BACKBONES = ['convnext_small', 'efficientnet_b0', 'resnet50']   # within each seed the slowest goes first
SEEDS     = (42, 43, 44)
HEAD      = ([256], 'relu')   # the P1 standard head, the same for every backbone
BASE_CONFIG = 'configs/p1_lr1e-4.yaml'

INPUT     = Path('/kaggle/input')
RUNS_DIR  = Path('/kaggle/working/p1d_smoke' if SMOKE else '/kaggle/working/runs')
CODE_DIR  = Path('/tmp/cdc-cp')
DATA_DIR  = Path('/tmp/DPD_Small')
CACHE_DIR = Path('/tmp/cache')

## 2. GPUs and code

In [ ]:
import os, shutil, subprocess, sys, zipfile, time
import torch

assert torch.cuda.is_available(), 'Settings > Accelerator > GPU T4 x2'
N_GPU = torch.cuda.device_count()
print(N_GPU, 'x', torch.cuda.get_device_name(0), '| torch', torch.__version__)
if N_GPU < 2:
    print('Only one GPU: runs will go one at a time (about twice as long). Pick GPU T4 x2 to run two at once.')

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
found = next((p.parent.parent for p in INPUT.rglob('cdccp/__init__.py')), None)
if found is not None:
    shutil.copytree(found, CODE_DIR)
else:
    code_zip = next(INPUT.rglob('cdc-cp_code*.zip'), None)
    assert code_zip, 'Add the code dataset (cdc-cp_code.zip) as an input'
    with zipfile.ZipFile(code_zip) as z:
        z.extractall('/tmp')
assert (CODE_DIR / 'scripts' / 'calib_board.py').is_file(), 'old code dataset: upload the new cdc-cp_code.zip'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
from cdccp.models import BACKBONES as KNOWN
assert set(BACKBONES) <= set(KNOWN), f'code does not know {set(BACKBONES) - set(KNOWN)}: upload the new code zip'
print('code from', found or code_zip)


def sh(*args):
    proc = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'exit code {proc.returncode}: {args}')

## 3. Training data (train CSV and its 8,000 images only)

In [ ]:
import hashlib
import pandas as pd

train_csv = next(p for p in INPUT.rglob('*') if p.name.lower() == 'dpd_small_train.csv')
cols = ['filename', 'crop_id', 'disease_id', 'composition_id', 'crop_name', 'disease_name', 'composition_name']
train = pd.read_csv(train_csv, header=None, names=cols)
assert len(train) == 8000, f'expected 8,000 rows, got {len(train)}'
digest = hashlib.sha256(train_csv.read_bytes()).hexdigest()
assert digest.startswith('9f1b729a'), f'train CSV hash {digest[:8]} does not match the split manifest'

first = train.filename.iloc[0]
image_src = next((d for d in [train_csv.parent, *train_csv.parent.parent.rglob('*')] if d.is_dir() and (d / first).is_file()), None)
assert image_src is not None, f'could not find {first} near {train_csv.parent}'
DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(train_csv, DATA_DIR / 'DPD_Small_train.csv')
link = DATA_DIR / 'DPD_Small_dataset'
if not link.exists():
    link.symlink_to(image_src, target_is_directory=True)
missing = [n for n in train.filename if not (link / n).is_file()]
assert not missing, f'{len(missing)} images missing, e.g. {missing[:3]}'
os.environ['DPD_ROOT'] = str(DATA_DIR)
print(f'8,000 training images linked from {image_src}')

## 4. Restore finished runs from earlier outputs

In [ ]:
RUNS_DIR.mkdir(parents=True, exist_ok=True)
PLANNED = {f'sweep_{b}_h256_seed{s}' for b in BACKBONES for s in SEEDS}
restored = []
if not SMOKE:
    for last in INPUT.rglob('sweep_*_seed*/last.pt'):
        src, dst = last.parent, RUNS_DIR / last.parent.name
        if src.name not in PLANNED:
            continue
        old = dst / 'last.pt'
        if not old.exists() or torch.load(old, map_location='cpu', weights_only=False)['applied'] < torch.load(last, map_location='cpu', weights_only=False)['applied']:
            shutil.rmtree(dst, ignore_errors=True)
            shutil.copytree(src, dst)
            restored.append(dst.name)
print('restored:', sorted(restored) or 'nothing')

## 5. Image cache (local disk, a few minutes)

In [ ]:
sh(sys.executable, 'scripts/build_cache.py', '--verify', '--workers', 8, f'paths.cache_dir={CACHE_DIR}')

## 6. Run queue
Each GPU takes the next waiting run. Only the last 3 epoch checkpoints of a run are kept.

In [ ]:
from cdccp.selection import pick_backbone

COMMON = [f'paths.runs_dir={RUNS_DIR}', f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}']


def run_name(backbone, seed):
    return f'sweep_{backbone}_h256_seed{seed}'


def job(backbone, seed):
    hidden, act = HEAD
    args = [sys.executable, 'scripts/train.py', BASE_CONFIG, *COMMON, f'name=sweep_{backbone}_h256', f'model.backbone={backbone}',
            f'model.hidden={hidden}', f'model.activation={act}', f'seed={seed}', '--resume']
    if SMOKE:
        args += ['--max-epochs', '1']
    return run_name(backbone, seed), args


def prune(run_dir, keep=3):
    pts = sorted(run_dir.glob('epoch_*.pt'))
    for p in pts[:-keep]:
        p.unlink()


def run_queue(jobs):
    """Run (name, args) jobs, one per GPU at a time. A failed run is reported and left out of the ranking."""
    waiting, running, failed = list(jobs), {}, []
    print('queue:', [n for n, _ in waiting], flush=True)
    while waiting or running:
        for gpu in range(max(N_GPU, 1)):
            if gpu not in running and waiting:
                name, args = waiting.pop(0)
                log = open(RUNS_DIR / f'{name}.log', 'a')
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
                p = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
                running[gpu] = (name, p, log, 0, time.time())
                print(f'[gpu {gpu}] start {name}', flush=True)
        time.sleep(20)
        for gpu, (name, p, log, seen, t0) in list(running.items()):
            log.flush()
            lines = (RUNS_DIR / f'{name}.log').read_text(errors='replace').splitlines()
            for line in lines[seen:]:
                if line.startswith(('epoch', 'done', 'Traceback')) or 'Error' in line:
                    print(f'[gpu {gpu}] {name}: {line}', flush=True)
            running[gpu] = (name, p, log, len(lines), t0)
            if p.poll() is not None:
                log.close()
                del running[gpu]
                mins = (time.time() - t0) / 60
                if p.returncode:
                    failed.append(name)
                    print(f'[gpu {gpu}] FAILED {name} after {mins:.0f} min; last lines:', *lines[-15:], sep='\n', flush=True)
                else:
                    print(f'[gpu {gpu}] finished {name} in {mins:.0f} min', flush=True)
                    if (RUNS_DIR / name).is_dir():
                        prune(RUNS_DIR / name)
    if failed:
        print(f'WARNING failed runs, left out of the ranking: {failed} (logs in {RUNS_DIR})', flush=True)
    return failed


def board(*patterns):
    sh(sys.executable, 'scripts/calib_board.py', RUNS_DIR, '--tol', TOL, '--match', *patterns)
    return pd.read_csv(RUNS_DIR / 'calib_board.csv')

## 7. Train 3 backbones x 3 seeds

In [ ]:
if SMOKE:
    FAILED = run_queue([job(b, 42) for b in BACKBONES])
    assert not FAILED, f'smoke runs failed: {FAILED}. Read their logs in {RUNS_DIR} before the full run.'
    print('SMOKE run finished: every backbone trains on this GPU. Set SMOKE = False and Save & Run All (Commit).')
else:
    FAILED = run_queue([job(b, s) for s in SEEDS for b in BACKBONES])

## 8. Pick the backbone
Saved as `runs/backbone_pick.md`, `calib_board.md` and `calib_board.csv` in this version's output.

In [ ]:
if not SMOKE:
    if FAILED:
        verdict = (f'NO PICK: runs failed: {FAILED}. Read their logs above. Then add this version as an input and Save & Run All again: '
                   'finished runs are skipped, failed ones resume.')
    else:
        final = board(*[f'sweep_{b}_h256_seed*' for b in BACKBONES])
        table, WINNER, SECOND, TIE, LIMIT = pick_backbone(final, BACKBONES, SEEDS)
        show = table.rename(columns={'unseen_disease': 'unseen disease (mean)', 'sd': 'SD over seeds', 'per_seed': f'per seed {SEEDS}',
                                     'H_keep': 'keep-seen H', 'seen_disease': 'seen disease'}).round(2)
        print(show.to_string())
        gap = table.unseen_disease.iloc[0] - table.unseen_disease.iloc[1]
        if TIE:
            verdict = (f'TIE: {WINNER} leads {SECOND} by {gap:.1f} points; a win needs more than {LIMIT:.1f} (seed spread plus the noise of 400 unseen images). '
                       'The data does not pick a winner. Decide on other grounds (seen disease, speed).')
        else:
            verdict = f'WINNER: {WINNER}, ahead of {SECOND} by {gap:.1f} points (a win needs more than {LIMIT:.1f}).'
        verdict = '```\n' + show.to_string() + '\n```\n\n' + verdict
    print(verdict)
    (RUNS_DIR / 'backbone_pick.md').write_text('# P1d backbone pick (development validation)\n\n' + verdict + '\n', encoding='utf-8')

## 9. Curves

In [ ]:
if not SMOKE and not FAILED:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    colours = dict(zip(BACKBONES, ['tab:blue', 'tab:orange', 'tab:green']))
    for b in BACKBONES:
        for s in SEEDS:
            h = pd.read_csv(RUNS_DIR / run_name(b, s) / 'history.csv')
            label = f'{b} s{s}'
            axes[0].plot(h.epoch, h.unseen_disease_top1, color=colours[b], label=label)
            axes[1].plot(h.epoch, h.seen_disease_top1, color=colours[b], label=label)
            axes[2].plot(h.epoch, h.val_ce_disease, color=colours[b], label=label)
    for ax, t in zip(axes, ['Unseen disease Top-1', 'Seen disease Top-1', 'Validation disease cross-entropy (seen)']):
        ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=6)
    plt.tight_layout(); plt.show()